<a href="https://colab.research.google.com/github/abi1264/test/blob/main/MRIwithADAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings

#supress all warnings
warnings.filterwarnings("ignore")

#define the path
train_dir='/content/drive/MyDrive/Research/Epic and CSCR hospital Dataset/train'
test_dir='/content/drive/MyDrive/Research/Epic and CSCR hospital Dataset/test'

#create an object of ImageDataGenerator for data augmentation and preprocessing
train_datagen=ImageDataGenerator(
    rescale=1./255,
    shear_range=0.4,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    rotation_range=90,
    horizontal_flip=True,
    fill_mode='nearest'
    validation_split=0.2
)

validation_datagen=ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
#No data augmentation for test data preprocessing
test_datagen=ImageDataGenerator(
    rescale=1./255
)

#Load the images and Labels ensuring grayscale image (1 color channel)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),   # resize images to 128 * 128
    color_mode='grayscale', #Load as grayscale
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)

validation_generator=validation_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)


test_generator=test_datagen.flow_from_directory(
    test_dir,
    target_size=(128,128),
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False,
    #when batch size is not given keras automatically takes 32 as default
)





In [ ]:
#Building our CNN model

from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import Flatten
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import BatchNormalization
import numpy as np
#Above imports can be written in single line also

#Define the CNN model
CNNadam=Sequential()

#Convolutional layer
CNNadam.add(Conv2D(256,(3,3),activation='relu',input_shape=(128,128,1)))
CNNadam.add(BatchNormalization())
CNNadam.add(MaxPooling2D(pool_size=(2,2)))

CNNadam.add(Conv2D(64,(3,3),activation='relu',input_shape=(128,128,1)))
CNNadam.add(BatchNormalization())
CNNadam.add(MaxPooling2D(pool_size=(2,2)))

CNNadam.add(Conv2D(256,(3,3),activation='relu',input_shape=(128,128,1)))
CNNadam.add(BatchNormalization())
CNNadam.add(MaxPooling2D(pool_size=(2,2)))

CNNadam.add(Conv2D(128,(3,3),activation='relu',input_shape=(128,128,1)))
CNNadam.add(BatchNormalization())
CNNadam.add(MaxPooling2D(pool_size=(2,2)))

CNNadam.add(Conv2D(64,(3,3),activation='relu',input_shape=(128,128,1)))
CNNadam.add(BatchNormalization())
CNNadam.add(MaxPooling2D(pool_size=(2,2)))

#Flattening the final 2D output matrix before feeding into the Dense layers
CNNadam.add(Flatten())

#Fully Connected Layers
CNNadam.add(Dense(128,activation='relu'))
CNNadam.add(BatchNormalization())
CNNadam.add(Dropout(0.2)) #Dropout to avoid overfitting
CNNadam.add(Dense(64,activation='relu'))
CNNadam.add(BatchNormalization())
CNNadam.add(Dropout(0.2))
CNNadam.add(Dense(32,activation='relu'))

#Output layer(adjust number of units to match the number of classes)
CNNadam.add(Dense(4,activation='softmax')) #assuming 4 classes in the dataset

#Compile the model
CNNadam.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy',metrics=['accuracy','precision','recall'])

#Model Summary
CNNadam.summary()






In [ ]:
start_time = time.time()

In [ ]:
#Train the model
history=CNNadam.fit(train_generator,batch_size=32,epochs=30,validation_data=validation_generator)


In [ ]:
end_time = time.time()

elapsed = end_time - start_time

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Training time: {hours}h {minutes}m {seconds:.2f}s")

In [ ]:
from keras.src.metrics.confusion_metrics import Precision
#Evaluate the model on the test data
loss1,accuracy1,precision1,recall1=CNNadam.evaluate(test_generator)

#print the accuracy and loss
print(f"Test Loss : {loss1}")
print(f"Accuracy :  {accuracy1}")
print(f"Precision : {precision1}")
print(f"Recall :{recall1}")





In [ ]:
CNNadam.save("brain_tumor_adam_optimizer.h5")

In [ ]:
CNNadam.save("brain_tumor_adam_optimizer.h5")

import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Adam Optimizer - Training and Validation Accuracy')
plt.legend()
plt.grid()
plt.show()


# Loss plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Adam Optimizer - Training and Validation Loss')
plt.legend()
plt.grid()
plt.show()